# Stroke — XAI extension

The reusable XAI framework is applied to the pre-specified class-balanced logistic-regression model using training-selected features. BMI imputation, feature selection and scaling are learned from the training data only.

The balanced model is specified for the extension independently of the held-out classifier ranking.

The framework generates explanations and assesses bootstrap-refit stability. The ordinary global permutation analysis uses ROC AUC, while stability uses negative log loss consistently across the four retrainable datasets. Results are exported to `../../results/stroke`.



The exported global SHAP analysis uses SHAP 0.51.0 `ExactExplainer`.


In [ ]:
import os

os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count() or 1))

import warnings
from tqdm import TqdmWarning

warnings.filterwarnings(
    "ignore",
    message="IProgress not found.*",
    category=TqdmWarning,
)

import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name != "xai_extension":
    raise RuntimeError(
        "Run this notebook with the working directory set to "
        "thesis/src/xai_extension. Current directory: "
        f"{NOTEBOOK_DIR}"
    )

PROJECT_ROOT = NOTEBOOK_DIR.parents[1]
FRAMEWORK_DIR = PROJECT_ROOT / "src" / "xai_framework"
if str(FRAMEWORK_DIR) not in sys.path:
    sys.path.insert(0, str(FRAMEWORK_DIR))

from run_pipeline import run_pipeline

file_path = "../../data/stroke/healthcare-dataset-stroke-data.csv"


## Data and model preparation


In [ ]:
df = pd.read_csv(file_path)
df = df[df['gender'] != 'Other'].copy()
df['age'] = df['age'].astype(int)

y = df['stroke'].copy()
X_raw = df.drop(columns=['stroke', 'id']).copy()

# Split before imputation, encoding, feature selection, and scaling.
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)

for part in (X_train_raw, X_test_raw):
    part['bmi_missing'] = part['bmi'].isna().astype(int)

train_bmi_mean = X_train_raw['bmi'].mean()
X_train_raw['bmi'] = X_train_raw['bmi'].fillna(train_bmi_mean)
X_test_raw['bmi'] = X_test_raw['bmi'].fillna(train_bmi_mean)

categorical_cols = [
    'gender', 'smoking_status', 'Residence_type',
    'work_type', 'ever_married'
]
X_train_encoded = pd.get_dummies(
    X_train_raw, columns=categorical_cols, drop_first=False
)
X_test_encoded = pd.get_dummies(
    X_test_raw, columns=categorical_cols, drop_first=False
).reindex(columns=X_train_encoded.columns, fill_value=0)

# Learn the correlation-based feature set from training data only.
training_for_selection = X_train_encoded.copy()
training_for_selection['stroke'] = y_train.to_numpy()
training_correlations = training_for_selection.corr(numeric_only=True)['stroke'].abs()
selected_features = training_correlations[
    (training_correlations > 0.1)
    & (training_correlations.index != 'stroke')
].index.tolist()

X_train_full = X_train_encoded
X_test_full = X_test_encoded
X_train_sel = X_train_encoded[selected_features]
X_test_sel = X_test_encoded[selected_features]

In [ ]:
scaler_full = StandardScaler()
X_train_full = pd.DataFrame(
    scaler_full.fit_transform(X_train_full),
    columns=X_train_encoded.columns, index=X_train_encoded.index
)
X_test_full = pd.DataFrame(
    scaler_full.transform(X_test_full),
    columns=X_train_encoded.columns, index=X_test_encoded.index
)

scaler_sel = StandardScaler()
X_train_sel = pd.DataFrame(
    scaler_sel.fit_transform(X_train_sel),
    columns=selected_features, index=X_train_encoded.index
)
X_test_sel = pd.DataFrame(
    scaler_sel.transform(X_test_sel),
    columns=selected_features, index=X_test_encoded.index
)

In [ ]:
# Pre-specified extension model. Logistic regression provides a stable probabilistic baseline for the highly imbalanced outcome and was not
# selected by inspecting the held-out test ranking.
selected_model_name = "Logistic Regression"
final_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42,
)
final_model.fit(X_train_sel, y_train)
print("Pre-specified XAI model:", selected_model_name)

Pre-specified XAI model: Logistic Regression


In [ ]:
feature_names = list(X_train_sel.columns)
best_model = final_model


def train_model_fn(X_bootstrap, y_bootstrap):
    """Refit the fixed Stroke estimator on one bootstrap training sample.

    Args:
        X_bootstrap: Bootstrap-resampled training feature matrix.
        y_bootstrap: Outcomes aligned with the bootstrap feature rows.

    Returns:
        A newly cloned and fitted estimator with the pre-specified Stroke
        configuration.
    """
    model = clone(best_model)
    model.fit(X_bootstrap, y_bootstrap)
    return model

## Execute the XAI pipeline


In [ ]:
PILOT = False

config = {
    "dataset_name": "stroke",

    "model": best_model,

    "X_train": X_train_sel,
    "X_test": X_test_sel,

    "y_train": y_train,
    "y_test": y_test,

    "feature_names": feature_names,

    # Repeated local explanations
    "local_cases_per_type": 2 if PILOT else 10,
    "lime_seeds": list(range(2 if PILOT else 20)),
    "lime_num_features": 5,
    "lime_stability_top_k": 3,

    "train_model_fn": train_model_fn,

    # Bootstrap-refit stability
    "stability_n_bootstrap": 10 if PILOT else 100,
    "stability_bootstrap_seed": 42,
    "stability_scoring": "neg_log_loss",
    "stability_n_repeats": 10,
    "stability_permutation_seed": 42,
    "stability_top_k": 3,

    # Ordinary global permutation analysis
    "scoring": "roc_auc",

    # Keep pilot results separate
    "output_dir": (
        "../../results/pilot_stability/stroke"
        if PILOT
        else "../../results/stroke"
    ),
}

experiment = run_pipeline(config)

Results exported to: ../../results/stroke


## XAI framework output sanity checks


In [ ]:
print("Dataset:", config["dataset_name"])
print("Model:", type(config["model"]).__name__)

print("X_train shape:", config["X_train"].shape)
print("X_test shape:", config["X_test"].shape)

print("y_train length:", len(config["y_train"]))
print("y_test length:", len(config["y_test"]))

print("Feature names:", len(config["feature_names"]))

Dataset: stroke
Model: LogisticRegression
X_train shape: (4087, 7)
X_test shape: (1022, 7)
y_train length: 4087
y_test length: 1022
Feature names: 7


### Input consistency


In [ ]:
assert config["X_train"].shape[1] == len(config["feature_names"])
assert config["X_test"].shape[1] == len(config["feature_names"])

assert len(config["X_train"]) == len(config["y_train"])
assert len(config["X_test"]) == len(config["y_test"])

print("Basic shape validation: OK")

Basic shape validation: OK


### Prediction-interface check


In [ ]:
model = config["model"]
X_test = config["X_test"]

if hasattr(model, "predict_proba"):
    probs = model.predict_proba(X_test)[:, 1]
    print("Probability range:", probs.min(), probs.max())

preds = model.predict(X_test)

print("Predictions:", np.unique(preds, return_counts=True))
print("Prediction test: OK")

Probability range: 0.0088262240727006 0.9864089793069091
Predictions: (array([0, 1]), array([741, 281]))
Prediction test: OK


### Framework results


In [ ]:
global_results = experiment.state.results["global"]
local_results = experiment.state.results["local"]
stability_results = experiment.state.results["stability"]

print(global_results.keys())
print(local_results.keys())

if stability_results is not None:
    print(stability_results.keys())
else:
    print("Stability: not available")

dict_keys(['native_importance', 'permutation_importance', 'shap_importance', 'shap_values', 'shap_raw_values', 'explainer', 'shap_explainer_class', 'rank_compare', 'rank_agreement'])
dict_keys(['cases', 'lime', 'lime_summary', 'lime_seed_stability'])
dict_keys(['raw', 'summary', 'rank_stability'])


### Global explanations


In [ ]:
native_df = global_results["native_importance"]
perm_df = global_results["permutation_importance"]
shap_df = global_results["shap_importance"]

print("Native:")
display(native_df.head())

print("Permutation:")
display(perm_df.head())

print("SHAP:")
display(shap_df.head())

Native:


,Feature,Importance,RawValue,Method
0,age,1.674441,1.674441,coefficients
4,bmi_missing,0.331027,0.331027,coefficients
3,avg_glucose_level,0.212911,0.212911,coefficients
1,hypertension,0.205457,0.205457,coefficients
2,heart_disease,0.059397,0.059397,coefficients


Permutation:


,Feature,Importance,Std
0,age,0.291633,0.028744
3,avg_glucose_level,0.004798,0.003867
4,bmi_missing,0.003608,0.003121
2,heart_disease,0.001931,0.000932
5,ever_married_No,-0.000357,0.000490


SHAP:


,Feature,MeanAbsSHAP,MeanSHAP
0,age,0.224165,-0.019177
1,avg_glucose_level,0.028314,-0.004800
2,hypertension,0.021344,0.001532
3,bmi_missing,0.016160,0.002403
4,heart_disease,0.003113,0.000240


### Feature coverage


In [ ]:
if native_df is not None:
    print("Native features:", len(native_df))

print("Permutation features:", len(perm_df))
print("SHAP features:", len(shap_df))
print("Expected features:", len(config["feature_names"]))

Native features: 7
Permutation features: 7
SHAP features: 7
Expected features: 7


### Numerical integrity


In [ ]:
print("Permutation NaNs:", perm_df.isna().sum().sum())
print("SHAP NaNs:", shap_df.isna().sum().sum())

if native_df is not None:
    print("Native NaNs:", native_df.isna().sum().sum())

Permutation NaNs: 0
SHAP NaNs: 0
Native NaNs: 0


### Rank agreement


In [ ]:
print(global_results["rank_agreement"])

display(global_results["rank_compare"].head())

{'Native_vs_SHAP': np.float64(0.8928571428571429), 'Native_vs_Permutation': np.float64(0.7387687194103585), 'SHAP_vs_Permutation': np.float64(0.6306562238868914)}


,Feature,Importance_x,RawValue,Method,NativeRank,MeanAbsSHAP,MeanSHAP,SHAPRank,Importance_y,Std,PermutationRank
0,age,1.674441,1.674441,coefficients,1.0,0.224165,-0.019177,1.0,0.291633,0.028744,1.0
1,bmi_missing,0.331027,0.331027,coefficients,2.0,0.016160,0.002403,4.0,0.003608,0.003121,3.0
2,avg_glucose_level,0.212911,0.212911,coefficients,3.0,0.028314,-0.004800,2.0,0.004798,0.003867,2.0
3,hypertension,0.205457,0.205457,coefficients,4.0,0.021344,0.001532,3.0,-0.000742,0.003242,7.0
4,heart_disease,0.059397,0.059397,coefficients,5.0,0.003113,0.000240,5.0,0.001931,0.000932,4.0


### Representative cases


In [ ]:
cases = local_results["cases"]

display(
    cases[
        ["CaseType", "y_true", "y_pred", "y_prob"]
    ]
)

,CaseType,y_true,y_pred,y_prob
207,TP,1,1,0.974578
231,TP,1,1,0.953740
880,TP,1,1,0.950020
866,TP,1,1,0.949474
700,TP,1,1,0.933643
468,TP,1,1,0.888759
616,TP,1,1,0.887299
419,TP,1,1,0.873988
61,TP,1,1,0.872853
415,TP,1,1,0.868614


### Local explanations and seed stability


In [ ]:
lime_df = local_results["lime"]
lime_summary_df = local_results["lime_summary"]
lime_seed_stability_df = local_results["lime_seed_stability"]

print("Repeated LIME rows:", len(lime_df))
print("Cases by category:")
display(local_results["cases"].groupby("CaseType").size().rename("N cases"))
print("Seeds per case:")
display(lime_df.groupby(["CaseType", "CaseIndex"])["LIMESeed"].nunique().describe())
print("Aggregated LIME summary:")
display(lime_summary_df.head(20))
print("LIME seed-stability summary:")
display(
    lime_seed_stability_df.groupby("CaseType")[[
        "MeanPairwiseSpearman", "MeanTopKOverlap",
        "MeanRuleWeightSD", "MeanRuleSignConsistency"
    ]].mean().round(3)
)

Repeated LIME rows: 5000
Cases by category:


CaseType
Borderline    10
FN            10
FP            10
TN            10
TP            10
Name: N cases, dtype: int64

Seeds per case:


count    50.0
mean     20.0
std       0.0
min      20.0
25%      20.0
50%      20.0
75%      20.0
max      20.0
Name: LIMESeed, dtype: float64

Aggregated LIME summary:


,CaseType,Rule,MeanWeight,StdWeight,MeanAbsWeight,MedianWeight,PositiveWeightFraction,NExplanations,NCases,NSeeds,SelectionFrequency
0,Borderline,age > 0.78,0.512434,0.003697,0.512434,0.512307,1.0,100,5,20,0.500
1,Borderline,bmi_missing > -0.20,0.255580,0.023929,0.255580,0.259312,1.0,20,1,20,0.100
2,Borderline,bmi_missing <= -0.20,-0.250443,0.017302,0.250443,-0.249216,0.0,180,9,20,0.900
3,Borderline,-0.81 < age <= 0.08,-0.212965,0.005069,0.212965,-0.213287,0.0,20,1,20,0.100
4,Borderline,0.08 < age <= 0.78,0.110999,0.004776,0.110999,0.110832,1.0,80,4,20,0.400
5,Borderline,hypertension > -0.32,0.098991,0.011885,0.098991,0.094767,1.0,20,1,20,0.100
6,Borderline,hypertension <= -0.32,-0.096516,0.010468,0.096516,-0.095689,0.0,180,9,20,0.900
7,Borderline,avg_glucose_level > 0.18,0.067271,0.008428,0.067271,0.067527,1.0,120,6,20,0.600
8,Borderline,heart_disease <= -0.25,-0.039805,0.011603,0.039805,-0.038654,0.0,177,10,20,0.885
9,Borderline,avg_glucose_level <= -0.64,-0.038525,0.004382,0.038525,-0.038812,0.0,40,2,20,0.200


LIME seed-stability summary:


,MeanPairwiseSpearman,MeanTopKOverlap,MeanRuleWeightSD,MeanRuleSignConsistency
CaseType,,,,
Borderline,0.968,0.99,0.009,0.788
FN,0.935,1.00,0.011,0.726
FP,0.978,1.00,0.008,0.845
TN,0.971,1.00,0.009,0.840
TP,0.991,1.00,0.008,0.876


### Bootstrap-refit stability


In [ ]:
if stability_results is not None:
    display(stability_results["raw"].head())
    print(stability_results["summary"])

,Feature,Importance,ImportanceStd,Bootstrap,BootstrapSampleSeed,Rank,InTopK
0,age,0.058329,0.008239,1,383329927,1.0,True
1,hypertension,-0.021496,0.002750,1,383329927,7.0,False
2,heart_disease,0.001069,0.000322,1,383329927,2.0,True
3,avg_glucose_level,-0.014043,0.002458,1,383329927,6.0,False
4,bmi_missing,-0.005086,0.005752,1,383329927,5.0,False


             Feature  MeanImportance  StdImportance  MeanRank   StdRank  \
0                age        0.054034       0.027472     1.040  0.281411   
3    ever_married_No        0.002642       0.009513     2.805  1.098426   
4   ever_married_Yes        0.002642       0.009513     3.355  1.023203   
5      heart_disease       -0.001580       0.002004     3.560  0.977732   
2        bmi_missing       -0.004488       0.001271     4.390  1.053086   
1  avg_glucose_level       -0.015348       0.004978     6.300  0.674200   
6       hypertension       -0.018879       0.006202     6.550  0.641573   

   TopKFrequency  
0           1.00  
3           0.76  
4           0.72  
5           0.28  
2           0.24  
1           0.00  
6           0.00  


### Exported artefacts


In [ ]:
out_dir = Path(config["output_dir"]).resolve()

print("Output directory:", out_dir.relative_to(PROJECT_ROOT))

for path in sorted(out_dir.rglob("*")):
    if path.is_file():
        print(path.relative_to(out_dir))

Output directory: results/stroke
bundle\metadata.json
bundle\model.pkl
bundle\X_test.csv
bundle\X_train.csv
bundle\y_test.csv
bundle\y_train.csv
execution_times.json
lime_explanations.csv
lime_seed_stability.csv
lime_summary.csv
native_importance.csv
permutation_importance.csv
rank_agreement\Native_vs_Permutation.json
rank_agreement\Native_vs_SHAP.json
rank_agreement\SHAP_vs_Permutation.json
rank_compare.csv
representative_cases.csv
shap_importance.csv
stability.csv
stability_rank.csv
stability_summary.csv


In [ ]:
print(
    experiment.state.results["local"]["cases"].columns.tolist()
)

display(
    experiment.state.results["local"]["cases"]
)

['age', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi_missing', 'ever_married_No', 'ever_married_Yes', 'y_true', 'y_pred', 'y_prob', 'distance', 'CaseType', 'CaseIndex', 'CaseRank']


,age,hypertension,heart_disease,avg_glucose_level,bmi_missing,ever_married_No,ever_married_Yes,y_true,y_pred,y_prob,distance,CaseType,CaseIndex,CaseRank
207,1.533528,-0.323411,4.076319,2.550736,4.906488,-0.717905,0.717905,1,1,0.974578,0.474578,TP,207,1
231,1.665909,3.092042,4.076319,3.245812,-0.203812,-0.717905,0.717905,1,1,0.953740,0.453740,TP,231,2
880,1.224639,3.092042,-0.245319,-0.405071,4.906488,-0.717905,0.717905,1,1,0.950020,0.450020,TP,880,3
866,1.710036,3.092042,4.076319,2.463432,-0.203812,-0.717905,0.717905,1,1,0.949474,0.449474,TP,866,4
700,1.621782,3.092042,-0.245319,3.004047,-0.203812,-0.717905,0.717905,1,1,0.933643,0.433643,TP,700,5
468,1.533528,3.092042,-0.245319,0.551694,-0.203812,1.392942,-1.392942,1,1,0.888759,0.388759,TP,468,6
616,1.577655,-0.323411,4.076319,2.225920,-0.203812,1.392942,-1.392942,1,1,0.887299,0.387299,TP,616,7
419,1.489401,-0.323411,4.076319,2.813097,-0.203812,-0.717905,0.717905,1,1,0.873988,0.373988,TP,419,8
61,1.665909,3.092042,-0.245319,-0.713546,-0.203812,-0.717905,0.717905,1,1,0.872853,0.372853,TP,61,9
415,1.312893,3.092042,-0.245319,1.885881,-0.203812,-0.717905,0.717905,1,1,0.868614,0.368614,TP,415,10


In [ ]:
assert "CaseIndex" in experiment.state.results["local"]["cases"].columns
print("Stroke CaseIndex exists in memory.")

Stroke CaseIndex exists in memory.


In [ ]:
stroke_cases = pd.read_csv(
    Path("../../results/stroke/representative_cases.csv")
)

print(stroke_cases.columns.tolist())
display(stroke_cases)

['age', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi_missing', 'ever_married_No', 'ever_married_Yes', 'y_true', 'y_pred', 'y_prob', 'distance', 'CaseType', 'CaseIndex', 'CaseRank']

,age,hypertension,heart_disease,avg_glucose_level,bmi_missing,ever_married_No,ever_married_Yes,y_true,y_pred,y_prob,distance,CaseType,CaseIndex,CaseRank
0,1.533528,-0.323411,4.076319,2.550736,4.906488,-0.717905,0.717905,1,1,0.974578,0.474578,TP,207,1
1,1.665909,3.092042,4.076319,3.245812,-0.203812,-0.717905,0.717905,1,1,0.953740,0.453740,TP,231,2
2,1.224639,3.092042,-0.245319,-0.405071,4.906488,-0.717905,0.717905,1,1,0.950020,0.450020,TP,880,3
3,1.710036,3.092042,4.076319,2.463432,-0.203812,-0.717905,0.717905,1,1,0.949474,0.449474,TP,866,4
4,1.621782,3.092042,-0.245319,3.004047,-0.203812,-0.717905,0.717905,1,1,0.933643,0.433643,TP,700,5
5,1.533528,3.092042,-0.245319,0.551694,-0.203812,1.392942,-1.392942,1,1,0.888759,0.388759,TP,468,6
6,1.577655,-0.323411,4.076319,2.225920,-0.203812,1.392942,-1.392942,1,1,0.887299,0.387299,TP,616,7
7,1.489401,-0.323411,4.076319,2.813097,-0.203812,-0.717905,0.717905,1,1,0.873988,0.373988,TP,419,8
8,1.665909,3.092042,-0.245319,-0.713546,-0.203812,-0.717905,0.717905,1,1,0.872853,0.372853,TP,61,9
9,1.312893,3.092042,-0.245319,1.885881,-0.203812,-0.717905,0.717905,1,1,0.868614,0.368614,TP,415,10


In [ ]:
results_dir = Path(config["output_dir"]).resolve()
stability_file = results_dir / "stability.csv"

print("Reading:", stability_file)

stability = pd.read_csv(stability_file)

importance_by_bootstrap = stability.pivot(
    index="Bootstrap",
    columns="Feature",
    values="Importance",
)

print("Bootstrap refits:", len(importance_by_bootstrap))
print(
    "Distinct importance profiles:",
    len(importance_by_bootstrap.drop_duplicates()),
)

display(importance_by_bootstrap)

Reading: results/stroke\stability.csv


Bootstrap refits: 100
Distinct importance profiles: 100


Feature,age,avg_glucose_level,bmi_missing,ever_married_No,ever_married_Yes,heart_disease,hypertension
Bootstrap,,,,,,,
1,0.058329,-0.014043,-0.005086,0.000526,0.000526,0.001069,-0.021496
2,0.088180,-0.007419,-0.003939,0.012882,0.012882,0.001673,-0.026341
3,0.043643,-0.013844,-0.004999,-0.001822,-0.001822,-0.001824,-0.027125
4,0.056060,-0.015531,-0.001915,0.002172,0.002172,-0.006058,-0.010627
5,0.013989,-0.017990,-0.005260,-0.009875,-0.009875,-0.001819,-0.028486
...,...,...,...,...,...,...,...
96,0.151378,-0.018906,-0.006949,0.033013,0.033013,-0.000737,-0.018150
97,0.073187,-0.023741,-0.005690,0.010226,0.010226,-0.000598,-0.010652
98,0.093484,-0.015754,-0.004830,0.015038,0.015038,-0.000325,-0.012135


## Interpretation

Age was the most persistent global feature, while encoded marital-status terms also appeared frequently in bootstrap top-three rankings. The `bmi_missing` indicator shows that the fitted model uses information associated with the measurement process; it is not evidence that missing BMI has a biological effect. Confident errors in the local analysis demonstrate why explanation coherence must be interpreted alongside observed outcomes.
